In [8]:
import os
import shutil
from pathlib import Path, PurePosixPath
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi, hf_hub_download

current_dir = Path.cwd().resolve()
search_dirs = [current_dir, *current_dir.parents]
PROJECT_ROOT = next((path for path in search_dirs if (path / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project root.")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw_videos_v2"
PROCESSED_DIR = DATA_DIR / "processed"
FRAMES_DIR = DATA_DIR / "frames_v2"
POSITIVE_DIR = RAW_DIR / "positive"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_DIR.mkdir(parents=True, exist_ok=True)
POSITIVE_DIR.mkdir(parents=True, exist_ok=True)

def relative_path(path):
    return Path(os.path.relpath(path, Path.cwd())).as_posix()

def find_mp4_files(folder):
    folder = Path(folder)
    found = []
    if not folder.exists():
        return found
    for root, dirs, files in os.walk(folder):
        dirs[:] = [name for name in dirs if name.lower() != ".cache"]
        for name in files:
            if name.lower().endswith(".mp4"):
                found.append(Path(root) / name)
    return sorted(found)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\Afrano\Documents\Quera\my_project\incident-sentiment-analyzer-20261029T081712Z-1-001\incident-sentiment-analyzer-20261029T081712Z-1-001\incident-sentiment-analyzer\notebooks


In [9]:
api = HfApi()
print("HF API initialized.")

HF API initialized.


In [13]:
import os

def verify_video(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        return None
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    if fps <= 0 or frame_count < 5:
        return None
    return True

print(f"Searching in OUTPUT_DIR: {OUTPUT_DIR}")

all_files_in_raw = []
for root, dirs, files in os.walk(OUTPUT_DIR):
    for file in files:
        if file.endswith(".mp4"):
            all_files_in_raw.append(Path(root) / file)

print(f"Found {len(all_files_in_raw)} MP4 files total.")

negative_candidates = []
seen_names = set()

for path in all_files_in_raw:
    parts = [p.lower() for p in path.parts]
    if "positive" not in parts and ".cache" not in parts:
        if path.stem not in seen_names:
            negative_candidates.append(path)
            seen_names.add(path.stem)

print(f"Found {len(negative_candidates)} potential negative videos.")

valid_negative_paths = []
for path in tqdm(negative_candidates, desc="Verifying negatives"):
    if verify_video(path):
        valid_negative_paths.append(path)

if len(valid_negative_paths) < 75:
    raise RuntimeError(f"Found {len(valid_negative_paths)} valid negatives, need 75. Checked {len(negative_candidates)} candidates.")

valid_negative_paths = sorted(valid_negative_paths, key=lambda p: p.name)[:75]
print(f"Selected {len(valid_negative_paths)} negative videos successfully.")

Searching in OUTPUT_DIR: c:\Users\Afrano\Documents\Quera\my_project\incident-sentiment-analyzer-20261029T081712Z-1-001\incident-sentiment-analyzer-20261029T081712Z-1-001\incident-sentiment-analyzer\data\raw_videos_v2
Found 233 MP4 files total.
Found 200 potential negative videos.


Verifying negatives:   0%|          | 0/200 [00:00<?, ?it/s]

Verifying negatives: 100%|██████████| 200/200 [00:06<00:00, 32.78it/s]

Selected 75 negative videos successfully.


In [14]:
# Cell 4: Select and download 75 positive collision videos from Hugging Face
import os
from huggingface_hub import hf_hub_download

all_files = api.list_repo_files(
    repo_id="nexar-ai/nexar_collision_prediction",
    repo_type="dataset"
)

# Filter for positive collision videos
positive_files = [f for f in all_files if f.endswith(".mp4") and ("positive" in f.lower() or "collision" in f.lower())]

# If folder structure uses a different path, fallback to any file containing positive
if not positive_files:
    positive_files = [f for f in all_files if f.endswith(".mp4")]

positive_files = positive_files[:75]
print(f"Total positive videos selected for download: {len(positive_files)}")

downloaded_positives = []
failed_positives = []

for filename in positive_files:
    try:
        # Save positive videos into a 'positive' subfolder inside OUTPUT_DIR
        local_sub_dir = OUTPUT_DIR / "positive"
        local_sub_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"Downloading positive: {filename}")
        path = hf_hub_download(
            repo_id="nexar-ai/nexar_collision_prediction",
            filename=filename,
            repo_type="dataset",
            token=HF_TOKEN,
            local_dir=local_sub_dir
        )
        downloaded_positives.append(path)
    except Exception as e:
        failed_positives.append((filename, str(e)))
        print(f"Failed to download {filename}: {e}")

print(f"\nPositive Download Done! Downloaded: {len(downloaded_positives)}, Failed: {len(failed_positives)}")

Total positive videos selected for download: 75

Positive Download Done! Downloaded: 75, Failed: 0


In [15]:
# Cell 5: Create and save valid_videos.csv metadata
import os
import pandas as pd
import cv2
from tqdm import tqdm
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Collect all positive mp4 files
positive_video_paths = []
pos_dir = OUTPUT_DIR / "positive"
for root, dirs, files in os.walk(pos_dir):
    for f in files:
        if f.endswith(".mp4"):
            positive_video_paths.append(Path(root) / f)

positive_video_paths = sorted(positive_video_paths, key=lambda p: p.name)[:75]

records = []

# Process 75 Negatives
for path in tqdm(valid_negative_paths, desc="Metadata for negatives"):
    cap = cv2.VideoCapture(str(path))
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    records.append({
        "video_id": path.stem,
        "video_path": str(path),
        "label": "negative",
        "fps": fps,
        "frame_count": frame_count,
        "duration": frame_count / fps if fps > 0 else 0.0,
        "width": width,
        "height": height
    })

# Process 75 Positives
for path in tqdm(positive_video_paths, desc="Metadata for positives"):
    cap = cv2.VideoCapture(str(path))
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    records.append({
        "video_id": path.stem,
        "video_path": str(path),
        "label": "positive",
        "fps": fps,
        "frame_count": frame_count,
        "duration": frame_count / fps if fps > 0 else 0.0,
        "width": width,
        "height": height
    })

df_valid = pd.DataFrame(records)
valid_videos_path = PROCESSED_DIR / "valid_videos.csv"
df_valid.to_csv(valid_videos_path, index=False)

print(f"Saved {len(df_valid)} videos to {valid_videos_path}")
print(df_valid["label"].value_counts())

Metadata for positives: 100%|██████████| 75/75 [00:02<00:00, 37.08it/s]


Saved 150 videos to ..\data\processed\valid_videos.csv
label
negative    75
positive    75
Name: count, dtype: int64


In [16]:
# Cell 6: Stratified split into Train, Val, and Test
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df_valid,
    test_size=0.30,
    random_state=42,
    stratify=df_valid["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

df_splits = pd.concat([train_df, val_df, test_df], ignore_index=True)
splits_path = PROCESSED_DIR / "video_splits_v2.csv"
df_splits.to_csv(splits_path, index=False)

print(f"Total split videos: {len(df_splits)}")
print(pd.crosstab(df_splits["split"], df_splits["label"]))

Total split videos: 150
label  negative  positive
split                    
test         12        11
train        52        53
val          11        11


In [17]:
# Cell 7: Extract 8 uniform frames per video (224x224)
import numpy as np

FRAMES_DIR = Path("../data/frames_v2")
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

NUM_FRAMES = 8
TARGET_SIZE = (224, 224)

all_frames_metadata = []

for idx, row in tqdm(df_splits.iterrows(), total=len(df_splits), desc="Extracting frames"):
    video_path = Path(row["video_path"])
    video_id = row["video_id"]
    split = row["split"]
    label = row["label"]
    
    split_dir = FRAMES_DIR / split
    split_dir.mkdir(parents=True, exist_ok=True)
    
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames < NUM_FRAMES:
        indices = list(range(total_frames)) + [total_frames - 1] * (NUM_FRAMES - total_frames)
    else:
        indices = np.linspace(0, total_frames - 1, NUM_FRAMES, dtype=int)
        
    for frame_idx, target_fidx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(target_fidx))
        ret, frame = cap.read()
        if not ret or frame is None:
            continue
            
        frame_resized = cv2.resize(frame, TARGET_SIZE)
        frame_filename = f"{video_id}_frame_{frame_idx:02d}.jpg"
        frame_path = split_dir / frame_filename
        
        cv2.imwrite(str(frame_path), frame_resized)
        
        all_frames_metadata.append({
            "video_id": video_id,
            "frame_idx": frame_idx,
            "frame_path": str(frame_path),
            "label": label,
            "split": split
        })
        
    cap.release()

df_all_frames = pd.DataFrame(all_frames_metadata)
print(f"Total extracted frames: {len(df_all_frames)}")
print(df_all_frames["split"].value_counts())

Extracting frames: 100%|██████████| 150/150 [05:30<00:00,  2.20s/it]

Total extracted frames: 1200
split
train    840
test     184
val      176
Name: count, dtype: int64


In [20]:
# Cell 8: Save final CSV splits, solution file, and integrity assertions
import re

train_frames = df_all_frames[df_all_frames["split"] == "train"].copy()
val_frames   = df_all_frames[df_all_frames["split"] == "val"].copy()
test_frames  = df_all_frames[df_all_frames["split"] == "test"].copy()

train_frames.to_csv(PROCESSED_DIR / "train_frames_v2.csv", index=False)
val_frames.to_csv(PROCESSED_DIR / "val_frames_v2.csv", index=False)
test_frames.to_csv(PROCESSED_DIR / "test_frames_v2.csv", index=False)

# Create solution_v2.csv per test video (PDF requirement)
test_videos = test_frames.drop_duplicates(subset=["video_id"])[["video_id", "label"]].copy()
test_videos["id"] = test_videos["video_id"].astype(str)
assert test_videos["id"].str.isdigit().all() and (test_videos["id"].str.len() == 5).all()

test_videos["target"] = (test_videos["label"] == "positive").astype(int)
solution_df = test_videos[["id", "target"]].copy()
solution_df.to_csv(PROCESSED_DIR / "solution_v2.csv", index=False)

# Integrity assertions
assert len(train_frames) + len(val_frames) + len(test_frames) == 1200, "Frame count mismatch!"
assert not train_frames.isna().values.any(), "Missing values in train!"
assert not val_frames.isna().values.any(), "Missing values in val!"
assert not test_frames.isna().values.any(), "Missing values in test!"
assert solution_df["id"].is_unique, "Duplicate solution IDs!"
assert solution_df.isna().sum().sum() == 0, "Missing values in solution!"

for path_str in df_all_frames["frame_path"].sample(20, random_state=42):
    assert Path(path_str).exists(), f"Frame file does not exist: {path_str}"

print("All assertions passed successfully!")
print("Saved files:")
print(f" - {PROCESSED_DIR / 'train_frames_v2.csv'} ({len(train_frames)} rows)")
print(f" - {PROCESSED_DIR / 'val_frames_v2.csv'} ({len(val_frames)} rows)")
print(f" - {PROCESSED_DIR / 'test_frames_v2.csv'} ({len(test_frames)} rows)")
print(f" - {PROCESSED_DIR / 'solution_v2.csv'} ({len(solution_df)} rows - per video)")

print("\nTrain class distribution:")
print(train_frames["label"].value_counts())

All assertions passed successfully!
Saved files:
 - ..\data\processed\train_frames_v2.csv (840 rows)
 - ..\data\processed\val_frames_v2.csv (176 rows)
 - ..\data\processed\test_frames_v2.csv (184 rows)
 - ..\data\processed\solution_v2.csv (23 rows - per video)

Train class distribution:
label
positive    424
negative    416
Name: count, dtype: int64


In [19]:
print("Video IDs shared by different files:",
      (df_splits.groupby("video_id")["video_path"].nunique() > 1).sum())

print("Duplicate frame paths:",
      df_all_frames["frame_path"].duplicated().sum())

print("Frame paths assigned both labels:",
      (df_all_frames.groupby("frame_path")["label"].nunique() > 1).sum())

Video IDs shared by different files: 0
Duplicate frame paths: 0
Frame paths assigned both labels: 0
